# Automated Tumor Detection in Whole Slide Images: An End-to-End Deep Learning Pipeline

**A practical guide to building a supervised deep learning system for detecting breast cancer metastases in histopathology images, using the CAMELYON16 challenge dataset.**

---

## Table of Contents

- [1. The Problem: Pathologist Shortages and the Promise of Automation](#section-1)
- [2. The CAMELYON16 Challenge](#section-2)
- [3. Environment Setup](#section-3)
- [4. Understanding Whole Slide Images](#section-4)
- [5. Step 1 - Finding the Tissue](#section-5)
- [6. Step 2 - Parsing Tumour Annotations and the Four-Class Labelling Scheme](#section-6)
- [7. Step 3 - Building the Training Dataset](#section-7)
- [8. Step 4 - The Training Pipeline](#section-8)
- [9. Step 5 - Model Architecture](#section-9)
- [10. Step 6 - Training](#section-10)
- [11. Step 7 - Test Set Evaluation](#section-11)
- [12. Results](#section-12)
- [13. Investigating the Field Cancerisation Hypothesis](#section-13)
- [14. Lessons Learned](#section-14)
- [15. Conclusion](#section-15)

---

<h2 id="section-1">1. The Problem: Pathologist Shortages and the Promise of Automation</h2>

Diagnosing cancer from tissue biopsies remains one of the most critical and labour-intensive tasks in modern medicine. A pathologist examining a sentinel lymph node biopsy must scan an entire tissue section at high magnification, searching for clusters of cancer cells that may occupy only a tiny fraction of the slide. In busy services, pathologists may face large daily caseloads under considerable time pressure.

The numbers tell a sobering story. In the UK, the Royal College of Pathologists has warned of a [sustained workforce crisis](https://www.rcpath.org/discover-pathology/public-affairs/the-pathology-workforce.html), with vacancy rates exceeding 30% in some specialties. In the US, the situation is similar: an ageing workforce, rising case volumes, and growing molecular testing demands all strain an already stretched system.

Meanwhile, the digitisation of histopathology is accelerating. Whole Slide Imaging (WSI) scanners now capture tissue sections at resolutions exceeding 100,000 × 100,000 pixels in images that capture cellular-level detail across an entire tissue section. This creates an opportunity: if we can train machine learning models to analyse these images, we can augment pathologists' workflows, flag suspicious regions for closer review, and potentially catch metastases that might be missed under time pressure.

A more speculative research question is whether models can also detect subtle alterations in tumour-adjacent or normal-appearing tissue, potentially capturing signals linked to the tissue microenvironment (a phenomena known as **field cancerisation**). This would represent a qualitatively different capability: not just automating what humans already do, but finding signals that humans cannot reliably perceive.

In this post, we build an end-to-end pipeline from raw whole-slide images to trained classifiers, and test how far these ideas hold up in practice.


<h2 id="section-2">2. The CAMELYON16 Challenge</h2>

The [CAMELYON16 Grand Challenge](https://camelyon16.grand-challenge.org/) was organised in 2015–2016 by the International Symposium on Biomedical Imaging (ISBI) to benchmark automated detection of breast cancer metastases in whole slide images of sentinel lymph node biopsies.

The dataset consists of nearly **400 H&E-stained whole slide images** from two Dutch medical centres (Radboud UMC and University Medical Centre Utrecht):

| Split | Tumor slides | Normal slides | Total |
|-------|-------------|--------------|-------|
| Train | 110 | 160 | 270 |
| Test  | 49  | 80  | 129 |

Each tumor slide comes with XML annotation files containing polygon outlines of metastatic regions, hand-drawn by expert pathologists.

The winning team (Wang et al., 2016) achieved a slide-level AUC of 0.925 using a GoogLeNet-based patch classifier trained on millions of patches. Remarkably, when the system’s predictions were combined with a pathologist’s review, the [pathologist’s AUC increased from 0.966 to 0.995, corresponding to an approximately 85% reduction in error](https://arxiv.org/pdf/1606.05718)

Our approach follows the same fundamental strategy (patch-based classification) but with one important modification: we introduce a **four-class labelling scheme** that lets us ask more nuanced questions about what the model is capable of actually detecting.

<h2 id="section-3">3. Running the code</h2>

This article shows the full notebook narrative and code, but the project is structured as a modular repository rather than a single standalone notebook. To run it yourself, clone the repo and open the notebook in Google Colab or a local Jupyter environment.

Sections 3–6 can be run directly and download slides from Amazon S3 on demand. Sections 10–12 require the pre-generated patch dataset, which you can create using the code in Section 7 (~6–8 hours generation time on Colab) or replace with your own dataset paths.

> **Note**: Training dataset generation and Model training requires Colab Pro (High RAM) to avoid out-of-memory crashes. All other sections run on the free tier.



In [ ]:
# Mount Google Drive and navigate to project folder
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/new_work/Projects/Camelyon16/camelyon16-pathology

!apt-get install -y openslide-tools > /dev/null 2>&1
!pip install -q -r requirements.txt

import numpy as np
import matplotlib.pyplot as plt
import openslide

np.random.seed(42)

# Project imports
from config import DEFAULT_CONFIG
from src.data import list_s3_files, download_file_from_s3, cleanup_file
from src.data.tissue_mask import get_tissue_mask, compute_foreground_mask
from src.data.tumor_polygons import load_tumor_polygons, classify_patch
from src.data.patch_extraction import (
    sample_grid_coordinates, sample_coordinates_by_class,
    extract_patch, preprocess_patch
)
from src.visualisation import (
    visualise_tissue_outline, visualise_patches_grid,
    find_zoom_region_by_coords, find_dense_tissue_region
)

# Example slide used throughout the notebook's tumor-slide walkthrough.
# Change this in one place to explore a different annotated tumor slide.
EXAMPLE_TUMOR_SLIDE_ID = 'tumor_005'
EXAMPLE_TUMOR_SLIDE = f'{EXAMPLE_TUMOR_SLIDE_ID}.tif'
EXAMPLE_TUMOR_ANNOTATION = f'{EXAMPLE_TUMOR_SLIDE_ID}.xml'

print("All imports successful!")

<h2 id="section-4">4. Understanding Whole Slide Images</h2>

A WSI is not a regular image. At full resolution, a single slide can be 100,000 × 200,000 pixels (roughly **60 gigabytes** of uncompressed pixel data). You cannot load one into memory.

Instead, WSI formats (like TIFF) use a **pyramidal structure**: the same image stored at multiple resolutions. We use the [OpenSlide](https://openslide.org/) library to navigate this pyramid, reading small regions on demand without loading the entire file.

Let's download a single slide and explore its structure. 


In [ ]:
# List available slides from S3
all_slides = list_s3_files(DEFAULT_CONFIG.data.s3_images, '.tif')
normal_slides = sorted([f for f in all_slides if 'normal' in f.lower()])
tumor_slides = sorted([f for f in all_slides if 'tumor' in f.lower()])

print(f"Dataset: {len(normal_slides)} normal slides, {len(tumor_slides)} tumor slides")
print(f"\nExample normal slide: {normal_slides[0]}")
print(f"Example tumor slide: {tumor_slides[0]}")

In [ ]:
# Download one tumor slide to explore
slide_name = EXAMPLE_TUMOR_SLIDE
slide_path = download_file_from_s3(
    DEFAULT_CONFIG.data.s3_images, slide_name, '/tmp'
)
slide = openslide.OpenSlide(slide_path)

# Explore the pyramid structure
print(f"Slide: {slide_name}")
print(f"Dimensions (level 0): {slide.dimensions[0]:,} × {slide.dimensions[1]:,} pixels")
print(f"Number of levels: {slide.level_count}")
print(f"\nPyramid levels:")
for i in range(slide.level_count):
    w, h = slide.level_dimensions[i]
    ds = slide.level_downsamples[i]
    print(f"  Level {i}: {w:>7,} × {h:>7,}  (downsample: {ds:.1f}×)")

In [ ]:
# View the whole slide as a thumbnail
thumbnail = slide.get_thumbnail((800, 800))

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(thumbnail)
ax.set_title(f'{slide_name} — Thumbnail', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nThe thumbnail is {thumbnail.size[0]}×{thumbnail.size[1]} pixels.")
print(f"The actual slide is {slide.dimensions[0]:,}×{slide.dimensions[1]:,} pixels.")
print(f"That's a {slide.dimensions[0] // thumbnail.size[0]:,}× reduction!")


<h2 id="section-5">5. Step 1 — Finding the Tissue</h2>

Most of a WSI is empty white background (the glass slide). Before extracting patches, we need to find where the actual tissue is — but the approach matters as much as the result.

### Design Decision: Why Work at Thumbnail Resolution?

At full resolution, a single slide may be 100,000 × 200,000 pixels. Exhaustively scanning it pixel-by-pixel to locate tissue would take minutes per slide and require gigabytes of RAM just for I/O. It is also entirely unnecessary.

Tissue detection is a *coarse localisation problem*: we only need to know roughly where to look. A 512 × 512 thumbnail, generated in milliseconds by OpenSlide, contains more than enough information to answer that question.

Crucially, the tissue mask is **never fed to the classifier**. It only identifies candidate patch coordinates. Once we have those coordinates in thumbnail space, `get_scaling_factors()` maps them back to level-0 slide space using a simple ratio:

```
scale_x = slide_width  / mask_width
scale_y = slide_height / mask_height

thumbnail pixel (col, row)
    ↓  ×  scale_x,  ×  scale_y
patch centre (x, y) in level-0 slide coordinates
    ↓  openslide.read_region(location=(x - 112, y - 112), level=0, size=(224, 224))
full-resolution 224 × 224 patch
```

This detect-coarsely/extract-finely pattern is a standard idiom in computational pathology. `sample_grid_coordinates()` uses exactly this flow: iterate over mask pixels, check for tissue, scale to slide space, read the patch.

### Design Decision: Why So Many Cleanup Steps?

Grayscale thresholding works in H&E thumbnails because tissue stains pink/purple and is reliably darker than the white glass background. But a single threshold applied to a raw thumbnail produces a noisy mask riddled with artefacts. `compute_foreground_mask()` applies a deliberate chain of operations to handle each failure mode:

| Step | Function | What it removes |
|------|----------|-----------------|
| Threshold `< 180` | — | Selects all dark pixels |
| `remove_small_objects()` | `min_size=100` | Dust specks, fibres, staining dots |
| `clear_border()` | — | Any blob touching the image edge (scanner margins are often dark) |
| `remove_small_holes()` | `area_threshold=100` | Small voids within tissue blobs caused by pale staining |
| Border zeroing | `border_margin=5` | Residual edge artefacts that survive `clear_border()` |

`filter_valid_components()` then applies a second filtering pass over the remaining connected components using **two criteria**: minimum area *and* maximum aspect ratio. Size alone is not enough — a thin horizontal strip from scanner calibration can have a large pixel area but an aspect ratio of 10:1, making it clearly not tissue. The code rejects any blob exceeding `max_aspect_ratio=5.0`.

**Order matters.** `clear_border()` must run *before* `filter_valid_components()`. If a border artefact is not removed first, it may be joined to real tissue by a thin pixel bridge and survive as a huge elongated blob.

> **Teaching point:** Segmentation pipelines are rarely a single magical step. They are a chain of heuristics, each handling a specific failure mode. The code in `compute_foreground_mask()` and `filter_valid_components()` encodes practical experience with WSI artefacts — the kind of knowledge that never appears in papers.

### Under the Hood: Tissue Masking

The tissue masking logic is surprisingly simple. Here's what `compute_foreground_mask` does — just ~10 lines of core logic:


In [ ]:
# === What compute_foreground_mask does internally ===
# (This is the actual logic — we'll use the module version below for the pipeline)

from skimage.morphology import remove_small_objects, remove_small_holes
from skimage.segmentation import clear_border

# Step 1: Get a tiny thumbnail (512×512) from the gigapixel image
thumbnail_gray = slide.get_thumbnail((512, 512)).convert("L")
thumbnail_array = np.array(thumbnail_gray)

# Step 2: Simple brightness threshold
# Tissue is darker than the white glass background
threshold = 220  # pixels darker than this are tissue
raw_mask = thumbnail_array < threshold

# Step 3: Morphological cleanup
cleaned = remove_small_objects(raw_mask, min_size=500)   # remove dust specks
cleaned = clear_border(cleaned)                           # remove edge artifacts
cleaned = remove_small_holes(cleaned, area_threshold=1000) # fill holes in tissue

print(f"Raw mask pixels:     {raw_mask.sum():,}")
print(f"After cleanup:       {cleaned.sum():,}")
print(f"Removed {raw_mask.sum() - cleaned.sum():,} artifact pixels")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(thumbnail_array, cmap='gray')
axes[0].set_title('Grayscale thumbnail')
axes[1].imshow(raw_mask, cmap='gray')
axes[1].set_title(f'After threshold (<{threshold})')
axes[2].imshow(cleaned, cmap='gray')
axes[2].set_title('After morphological cleanup')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Generate tissue mask
mask = compute_foreground_mask(slide)

print(f"Mask shape: {mask.shape}")
print(f"Tissue coverage: {mask.sum() / mask.size:.1%}")

# Visualise: original thumbnail vs. detected tissue
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original
axes[0].imshow(thumbnail)
axes[0].set_title('Original Slide', fontsize=13)
axes[0].axis('off')

# Binary mask
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Tissue Mask', fontsize=13)
axes[1].axis('off')

# Overlay: tissue outline on slide
axes[2].imshow(thumbnail)
# Resize mask to match thumbnail
from PIL import Image
mask_resized = np.array(
    Image.fromarray(mask.astype(np.uint8) * 255).resize(thumbnail.size, Image.NEAREST)
) > 127
axes[2].contour(mask_resized.astype(float), levels=[0.5], colors='lime', linewidths=1.5)
axes[2].set_title('Tissue Detection Overlay', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.show()


<h2 id="section-6">6. Step 2 — Parsing Tumour Annotations and the Four-Class Labelling Scheme</h2>

For tumour slides, CAMELYON16 provides XML files with polygon annotations tracing the tumour boundaries. We parse these into [Shapely](https://shapely.readthedocs.io/) geometry objects, which lets us compute precise overlap between any patch and the annotated tumour regions.

### Design Decision: Labels Come from Geometry, Not Slide Identity

The label assigned to each patch is **not** simply determined by which slide it came from. It is determined by computing the exact geometric overlap between the patch (a 224 × 224 square in slide coordinates) and the annotated tumour polygons.

The process inside `classify_patch()`:

1. The patch centre `(x, y)` is converted to a `shapely.box` — a rectangle in level-0 slide coordinate space.
2. `calculate_tumor_overlap()` iterates over all tumour polygons, summing the intersection area between the patch box and each polygon.
3. That total is divided by the patch area (224²) to get an overlap fraction in [0, 1].
4. Threshold rules from `config.py` assign the class label:

```python
if overlap < zero_tolerance:       label = 1  # Normal tissue (tumour slide)
elif overlap < tumor_threshold:    label = 2  # Boundary  (default: overlap < 0.50)
else:                              label = 3  # Pure tumour
```

"Boundary" is not a vague visual impression — it is a formal geometric category defined by `boundary_threshold = 0.01` and `tumor_threshold = 0.50` in `config.py`. This makes the labelling reproducible and auditable. Class 0 (`normal_from_normal`) is handled separately: it applies to patches from slides that have no annotation file at all.

### Design Decision: Repairing Invalid Polygon Geometry

Real-world annotation data is messy. XML files from CAMELYON16 can contain polygons with self-intersections or degenerate geometry — artefacts of how pathologists draw region outlines by hand. Rather than silently discarding these polygons (which would lose tumour annotations and bias the dataset), `load_tumor_polygons()` calls Shapely's `make_valid()` to repair them.

Repaired geometry may return a `MultiPolygon` when a self-intersecting region splits into disconnected parts, so the code handles both `Polygon` and `MultiPolygon` outputs:

```python
fixed = make_valid(polygon)
if hasattr(fixed, 'geoms'):          # MultiPolygon
    for geom in fixed.geoms:
        if isinstance(geom, Polygon): polygons.append(geom)
elif isinstance(fixed, Polygon):     # Single repaired polygon
    polygons.append(fixed)
```

> **Teaching point:** ML pipelines fail as often on input geometry as on model architecture. Discarding malformed annotations without logging them is a subtle form of data corruption — you would be training on a systematically incomplete view of the tumour regions.

### The Four-Class Scheme

Rather than a simple binary "normal vs. tumour" label, we classify each patch into **four categories** based on its spatial relationship to the tumour:

| Class | Name | Description |
|-------|------|-------------|
| 0 | `normal_from_normal` | Normal tissue from a slide with no tumour at all |
| 1 | `normal_from_tumor` | Normal-looking tissue on a slide that also contains tumour |
| 2 | `boundary_tumor` | Tissue at the tumour margin (partial overlap with annotations) |
| 3 | `pure_tumor` | Tissue fully within annotated tumour regions |

Keeping classes 0 and 1 separate — rather than merging them into a single "normal" class — is one of the most important scientific choices in this project. Class 1 patches may look identical to class 0 under the microscope, but they come from a slide where tumour is present elsewhere. If a model can distinguish them, it may be detecting **field cancerisation** — changes in apparently normal tissue driven by the nearby tumour microenvironment. This distinction powers Experiment 3.

> **Discussion prompt:** If a model successfully separates class 0 from class 1, what signal might it be using — cell morphology, staining intensity, tissue architecture, or something else? Could it be a biological signal at all, or might it reflect a dataset artefact such as staining batch effects between slides?

This scheme lets us ask more interesting questions than simple binary classification. In particular, **Class 1 vs. Class 0** tests the field cancerisation hypothesis: is there a detectable difference between "normal" tissue that happens to share a slide with a tumour and normal tissue from a completely healthy slide?

In [ ]:
# Load tumor annotations
xml_path = download_file_from_s3(
    DEFAULT_CONFIG.data.s3_annotations, EXAMPLE_TUMOR_ANNOTATION, '/tmp'
)
polygons = load_tumor_polygons(xml_path)
print(f"Loaded {len(polygons)} tumor polygons")
for i, p in enumerate(polygons):
    print(f"  Polygon {i}: area = {p.area:,.0f} pixels², "
          f"bounds = {tuple(int(x) for x in p.bounds)}")

In [ ]:
# Visualise tumor annotations overlaid on the tissue
visualise_tissue_outline(
    slide, mask,
    tumor_polygons=polygons,
    title='Tumor Annotations (red) with Tissue Outline (green)',
    figsize=(10, 10)
)


### Classifying Patches by Tumor Overlap

Each patch is classified by computing the **fractional overlap** between the 224×224 patch and the tumor polygons:
- **< 1% overlap** → Class 1 (normal tissue on tumor slide)
- **1–50% overlap** → Class 2 (boundary tissue)
- **≥ 50% overlap** → Class 3 (pure tumor)

The thresholds are configurable, but these defaults ensure clean separation between classes.


### Design Decision: Grid Sampling and Why Stride Is a Modelling Choice

`sample_grid_coordinates()` places a regular grid of candidate patch centres across the tissue mask, separated by a configurable `stride`. This approach is simple, reproducible, and fully auditable — every run with the same seed produces the same coordinates.

**Stride determines overlap and redundancy:**

| Stride | Overlap | Relative patch count |
|--------|---------|----------------------|
| 224 px (= patch size) | None — clean tiling | 1× |
| 112 px | 50% overlap | 4× |
| 56 px | 75% overlap | 16× |

Smaller stride captures more spatial context and improves coverage of small regions, but introduces **correlation** between nearby patches — consecutive patches at 112 px stride share 50% of their pixels. This matters for evaluation: correlated patches from the same region do not represent independent evidence, so effective sample size is overstated if patches are treated as i.i.d.

**Bounds checking:** `sample_grid_coordinates()` enforces that every patch centre is at least `patch_size // 2 = 112` pixels from each slide edge. This is necessary because patches are *centred* at `(x, y)` rather than anchored at the top-left corner — without this check, the code would attempt to read off the edge of the slide.

> **Student question:** What changes if stride goes from 224 to 112? You get 4× more patches — but adjacent pairs share 50% of pixels. Does this give 4× more *information*, or 4× more correlated samples? How would you test the difference empirically?

### Design Decision: Class-Specific Sampling Densities

Tumour boundary patches are rare and informationally dense. A slide may have only a thin rim of boundary tissue — sampling at a coarse uniform stride risks missing it almost entirely. `sample_coordinates_by_class()` addresses this by applying **class-specific strides**:

| Class | Default stride | Rationale |
|-------|----------------|-----------|
| Normal (class 1) | 224 px | Abundant everywhere; no overlap needed |
| Pure tumour (class 3) | 112 px | Moderate density for spatial coverage |
| Boundary (class 2) | 56 px | Dense sampling to capture the thin, rare tumour margin |

The implementation samples first at the finest stride (56 px) to classify every tissue coordinate, then subsamples each class independently. The keep ratio is `(target_stride / finest_stride)²` — **squared** because sampling occurs in 2D, not 1D. A stride ratio of 4 means keeping 1/16 of patches, not 1/4.

> **Why not sample uniformly everywhere?** Because uniform coarse sampling would dramatically undersample the boundary region — the class where the tumour/normal distinction is hardest and most valuable to learn.

### Under the Hood: Patch Classification with Shapely

How do we turn a patch coordinate into a class label? The key is computing the **geometric overlap** between the patch (a square) and the tumor polygons. Here's the core logic:


In [ ]:
# === What classify_patch does internally ===
from shapely.geometry import Polygon, box

# Pick an example coordinate near a tumor boundary
example_coords = coords_by_class_preview = sample_coordinates_by_class(slide_path, xml_path)

# Show the classification logic for 3 example patches (one per class)
for class_id in [1, 2, 3]:
    coords = example_coords.get(class_id, [])
    if not coords:
        continue
    x, y = coords[0]  # Take first patch of this class

    # Create a square box for the patch
    patch_size = 224
    half = patch_size // 2
    patch_box = box(x - half, y - half, x + half, y + half)
    patch_area = patch_size * patch_size

    # Compute intersection with ALL tumor polygons
    total_overlap = 0.0
    for polygon in polygons:
        if polygon.intersects(patch_box):
            intersection = polygon.intersection(patch_box)
            total_overlap += intersection.area

    overlap_fraction = min(total_overlap / patch_area, 1.0)

    # Apply classification thresholds
    if overlap_fraction < 0.01:
        label_name = "Class 1: Normal (tumor slide)"
    elif overlap_fraction < 0.50:
        label_name = "Class 2: Boundary"
    else:
        label_name = "Class 3: Pure Tumor"

    print(f"  Patch at ({x}, {y}): overlap = {overlap_fraction:.1%} → {label_name}")


In [ ]:
# Build a regular patch grid over tissue, then classify each patch.
# This preserves the spatial ordering of the grid for visualisation.
grid_stride = 224
coords = sample_grid_coordinates(slide, mask, patch_size=224, stride=grid_stride)

coords_by_class = {1: [], 2: [], 3: []}
for x, y in coords:
    label = classify_patch(x, y, polygons, patch_size=224)
    coords_by_class[label].append((x, y))

print(f"Regular grid stride: {grid_stride} pixels")
print(f"Total tissue patches on grid: {len(coords):,}")
for class_id, class_coords in sorted(coords_by_class.items()):
    class_names = {1: 'Normal (tumor slide)', 2: 'Boundary', 3: 'Pure Tumor'}
    print(f"  Class {class_id} ({class_names[class_id]}): {len(class_coords):,} patches")


In [ ]:
# Visualise the regular patch grid zoomed into the tumor region.
# Center the zoom on pure-tumor patches so the surrounding boundary and normal
# tissue appear in the same ordered grid, as in notebook 02 section 2.3.
zoom_coords = coords_by_class.get(3, []) or (coords_by_class.get(2, []) + coords_by_class.get(1, []))

if zoom_coords:
    zoom_region = find_zoom_region_by_coords(zoom_coords, region_size=10000)
    print(f"Zoom region: {zoom_region}")

    visualise_patches_grid(
        slide,
        coords_by_class,
        zoom_region=zoom_region,
        patch_size=224,
        class_colours={1: 'green', 2: 'orange', 3: 'red'},
        class_labels={
            1: 'Normal',
            2: 'Boundary',
            3: 'Pure Tumor'
        },
        title=f'Zoomed Tumor Region (Grid View) - {slide_name}',
        linewidth=1.5,
        figsize=(14, 12)
    )
else:
    print('No classified patch coordinates were found for visualisation.')


<h2 id="section-7">7. Step 3 — Building the Training Dataset</h2>

With our labelling scheme defined, we need to extract hundreds of thousands of patches from hundreds of slides and organise them into a training dataset. Several non-obvious design decisions shape how it works.

### Design Decision: Generate Four Classes, Collapse to Binary at Training Time

The dataset is generated with **four classes**, not two. The class labels are stored in every chunk alongside the patches. At training time, `run_binary_experiment()` remaps them to binary labels depending on the experiment being run.

This separates two concerns: *data collection* (done once, expensively) and *experimental question* (defined cheaply at training time). Generating four classes upfront enables all five binary experiments without re-running the ~6–8 hour extraction pipeline:

| Experiment | Negative class | Positive class |
|------------|----------------|----------------|
| 1 — Normal vs Any Tumour | `normal_from_normal` | classes 1, 2, 3 |
| 2 — Normal vs Pure Tumour | `normal_from_normal` | `pure_tumor` |
| 3 — Slide Context Detection | `normal_from_normal` | `normal_from_tumor` |
| 4 — Normal vs Actual Tumour | `normal_from_normal` | classes 2, 3 |
| 5 — Normal vs Boundary | `normal_from_normal` | `boundary_tumor` |

> **Teaching point:** Label design shapes which scientific questions you can ask. Collapsing to binary labels at collection time would have permanently closed off Experiment 3.

### Design Decision: Chunk by Slide, Verify Leakage Explicitly

We cannot hold all patches in memory simultaneously (≈ 130 GB uncompressed). Instead, patches are saved in compressed `.npz` chunks of ~1,000 patches each. Each chunk records:

- `X`: patch arrays `(N, 224, 224, 3)`
- `y`: class labels `(N,)`
- `slides`: source slide identifiers `(N,)` — **critical for leakage prevention**
- `coords`: original level-0 coordinates `(N, 2)` — for debugging and visualisation

**Chunking is not just a storage decision — it is a leakage-control mechanism.** WSIs produce many correlated patches. If patches from the same slide appear in both training and validation, the model can memorise slide-specific artefacts (staining quirks, tissue preparation differences) rather than learning genuine pathology. `FourClassGenerator` ensures each chunk contains patches from a single slide. Train/validation splits at the chunk level then guarantee no slide overlap by construction.

`verify_no_slide_leakage()` makes this guarantee explicit: after splitting, it reads the `slides` array from every chunk in both sets and raises a `ValueError` if any slide ID appears in both:

```python
train_slides = collect_slides(train_files)
val_slides   = collect_slides(val_files)
overlap = train_slides & val_slides
if overlap:
    raise ValueError(f"Slide leakage detected! {len(overlap)} slides in both sets.")
```

> **Teaching point:** Data leakage does not only happen at the row level in a dataframe. In WSI pipelines, it happens at the *slide* level — and it inflates validation metrics in ways that only become apparent on the held-out test set, sometimes by 10–15 AUC points.

### Stain Normalisation

H&E staining varies between labs, technicians, and even between slides from the same lab. This variation is irrelevant to the diagnostic task but can confuse a model. We apply **Macenko stain normalisation** during dataset generation: every patch is colour-adjusted to match a reference image, reducing spurious variation while preserving diagnostic features.

> **Dataset generation takes ~6–8 hours** on Colab and only needs to be run once. The generated dataset is saved to Google Drive for reuse. We provide the generator code below but skip execution — the pre-generated dataset is used for all subsequent steps.

In [ ]:
# === DATASET GENERATION (run once, then skip) ===
# This cell is provided for reference. The pre-generated dataset
# is loaded in the next section.

# from src.data.generator import generate_dataset, generate_test_dataset
#
# # Training dataset: ~100K patches per class
# generate_dataset(
#     class_targets={0: 100000, 1: 100000, 2: 100000, 3: 100000},
#     save_path='./data/camelyon16_4class_stain_normalised',
#     stain_normalise=True,
#     reference_image_path='./data/reference_patch.png'
# )
#
# # Test dataset: ~25K patches per class
# generate_test_dataset(
#     class_targets={0: 25000, 1: 25000, 2: 25000, 3: 25000},
#     save_path='./data/camelyon16_test_stain_normalised',
#     stain_normalise=True,
#     reference_image_path='./data/reference_patch.png'
# )

print("Dataset generation code shown above (pre-generated dataset used below)")


In [ ]:
# Dataset paths
TRAIN_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_4class_stain_normalised'
TEST_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_test_stain_normalised'

# Verify the dataset
import os
from pathlib import Path

class_names = {
    0: 'normal_from_normal',
    1: 'normal_from_tumor',
    2: 'boundary_tumor',
    3: 'pure_tumor'
}

print("=== Training Dataset ===")
total_train = 0
for class_id, name in class_names.items():
    class_dir = Path(TRAIN_PATH) / name
    chunks = list(class_dir.glob('*.npz'))
    if chunks:
        sample = np.load(str(chunks[0]))
        n_per_chunk = len(sample['X'])
        sample.close()
        total = len(chunks) * n_per_chunk
        total_train += total
        print(f"  {name}: {len(chunks)} chunks (~{total:,} patches)")

print(f"  Total: ~{total_train:,} patches")

print("\n=== Test Dataset ===")
total_test = 0
for class_id, name in class_names.items():
    class_dir = Path(TEST_PATH) / name
    chunks = list(class_dir.glob('*.npz'))
    if chunks:
        sample = np.load(str(chunks[0]))
        n_per_chunk = len(sample['X'])
        sample.close()
        total = len(chunks) * n_per_chunk
        total_test += total
        print(f"  {name}: {len(chunks)} chunks (~{total:,} patches)")

print(f"  Total: ~{total_test:,} patches")

<h2 id="section-8">8. Step 4 — The Training Pipeline</h2>

With our chunked dataset ready, we need a training pipeline that streams patches from chunks without loading everything into memory, remaps 4-class labels to binary for each experiment, prevents slide leakage between splits, and maintains class balance.

### Design Decision: Strictly Class-Balanced Batches

The training pipeline does not simply shuffle all patches together. `create_train_dataset()` builds **two separate class-specific streams** — one for the negative class, one for the positive — and forces every batch to draw equally from both:

```python
normal_ds = _create_single_class_dataset(normal_chunks, label=0)
tumor_ds  = _create_single_class_dataset(tumor_chunks,  label=1)

# Half the batch from each class, then concatenate and shuffle positions
dataset = tf.data.Dataset.zip((normal_ds.batch(half_batch), tumor_ds.batch(half_batch)))
          .map(lambda n, t: concat_and_shuffle(n, t))
```

This matters beyond simple class imbalance. With exact 50/50 balance enforced every batch:

1. **Stable BatchNorm statistics.** If a batch is dominated by one class, batch normalisation learns class-specific running statistics rather than tissue-general ones. This destabilises training.
2. **Consistent gradient updates.** The loss cannot be dominated by the majority class regardless of how chunks happen to be sampled.
3. **Position independence.** `_shuffle_batch()` randomly permutes sample order within each batch so the model cannot learn that "the first 16 samples are always normal".

### Design Decision: Validation Is Engineered for Stability, Not Just Correctness

Training and validation pipelines are built differently on purpose. `create_preloaded_val_dataset()` loads a fixed, class-balanced subset into memory once, interleaves classes at the sample level (N, T, N, T, …), and caches the result. The same data is seen every epoch.

This is a deliberate trade-off: slightly reduced coverage of the validation set in exchange for metrics that are directly comparable epoch-to-epoch.

Without this, validation accuracy oscillated significantly between epochs during development — not because the model was changing rapidly, but because a generator-based validation pipeline produces different batches on each call, introducing sampling noise into the metric itself. `diagnose_validation_stability()` was written specifically to detect and quantify this effect:

```python
# Runs model.evaluate() 5 times on the same dataset and reports variance.
# A stable pipeline has accuracy variance < 1%.
results = diagnose_validation_stability(model, val_dataset, num_checks=5)
```

> **Teaching point:** Students often assume validation is just another dataloader. In practice, how you build the validation pipeline directly affects whether your training curves are informative or misleading. Noisy validation metrics can cause early stopping to fire too soon, or make a plateau look like progress.

### Memory Management: The Key Engineering Challenge

Loading 400K patches would require ~130 GB of RAM. The solution is `tf.data.interleave()`, which reads from multiple chunk files simultaneously and yields patches on demand. A critical lesson from development: `num_parallel_calls=tf.data.AUTOTUNE` spawned too many workers and caused memory leaks. Setting `num_parallel_calls=min(2, cycle_length)` — an explicit, conservative limit — solved the issue.

### Preventing Slide Leakage

The pipeline splits chunks into train/val sets and then calls `verify_no_slide_leakage()`, which reads the `slides` array from every chunk in both sets and raises a `ValueError` on any overlap. Slide-aware splitting is enforced at the data-generation stage (`FourClassGenerator`) and verified again here at training time.

### Under the Hood: Streaming Chunks with `tf.data`

The training pipeline must feed ~400K patches to the model without loading them all into memory. Here's the core idea — each chunk file is read on demand using `tf.data.interleave`:

```python
# Simplified version of our chunk reading logic:

def read_chunk(file_path, label):
    with np.load(file_path, mmap_mode="r") as data:
        X = data['X']                     # Memory-mapped, not loaded yet
        idx = np.random.choice(len(X), max_patches, replace=False)
        patches = X[idx].astype(np.float32)  # Only NOW loaded into RAM

        # Normalise to [0, 1]
        if patches.max() > 1.5:
            patches /= 255.0
        patches = np.clip(patches, 0.0, 1.0)

        labels = np.full(len(patches), label, dtype=np.int32)
        return patches, labels

# tf.data.interleave reads from multiple chunks simultaneously,
# yielding a stream of patches without holding everything in memory:
dataset = file_dataset.interleave(
    read_chunk,
    cycle_length=4,           # Read 4 chunks at once
    num_parallel_calls=2,     # CRITICAL: not AUTOTUNE (causes memory leaks)
    deterministic=False       # Allow out-of-order for speed
)
```

**Class balancing** is enforced at the batch level: we create separate streams for each class, batch half from each, then concatenate. This guarantees exactly 50/50 class balance in every batch:

```python
normal_ds = create_class_stream(normal_chunks, label=0)
tumor_ds  = create_class_stream(tumor_chunks,  label=1)

# Each batch: 16 normal + 16 tumor = 32 balanced samples
balanced = tf.data.Dataset.zip((
    normal_ds.batch(16),
    tumor_ds.batch(16)
)).map(lambda n, t: concat(n, t))
```

**Slide leakage prevention**: after splitting chunks into train/val, we read the `slides` array from every chunk and verify zero overlap:

```python
train_slides = collect_slide_ids(train_chunks)
val_slides   = collect_slide_ids(val_chunks)
assert len(train_slides & val_slides) == 0, "Slide leakage detected!"
```


<h2 id="section-9">9. Step 5 — Model Architecture</h2>

We deliberately keep our CNN architectures simple and interpretable. The goal is to demonstrate the end-to-end pipeline, not to push state-of-the-art accuracy.

### Design Decision: Start Simple, Then Go Subtle

The architecture file provides two primary models. The choice between them encodes a hypothesis about what the task requires.

**`simple`** — for visually obvious class differences:
- Larger 5×5 kernels and stride 2 at every layer reduce spatial resolution aggressively.
- Fewer parameters (~66K), fast to train, low overfitting risk.
- Appropriate for Experiment 2 (Normal vs Pure Tumour), where tumour tissue has large-scale morphological differences.

**`subtle`** — for fine-grained tissue analysis:
- Small 3×3 kernels throughout capture finer local structure — individual cell boundaries, nuclear detail, gland architecture.
- The **first layer uses stride 1** rather than stride 2, preserving full spatial resolution for one extra stage before downsampling begins.
- Four convolutional blocks with increasing filter counts (32 → 64 → 128 → 256) give progressively more abstract representations.
- More parameters (~390K), heavier regularisation via dropout at each block.
- Appropriate for harder experiments where differences between classes are subtle or statistical rather than visually obvious.

```
simple:  Input → Conv(16, 5×5, s=2) → Conv(32, 5×5, s=2) → Conv(64, 5×5, s=2) → GAP → Dense(1)   [~66K params]

subtle:  Input → Conv(32, 3×3, s=1) → Conv(64, 3×3, s=2) → Conv(128, 3×3, s=2) → Conv(256, 3×3, s=2) → GAP → Dense(1)   [~390K params]
```

> **Teaching point:** Architecture choice encodes a hypothesis about what signal the model needs to detect. Using `subtle` for the field cancerisation experiment is a bet that the signal — if it exists — lives in fine local texture rather than coarse tissue morphology.

### Design Decision: Global Average Pooling Instead of Flattening

After the final convolutional layer, the feature map is 28 × 28 × 256 for `subtle`. Two options exist for converting this to a classification score:

- **Flatten** → a vector of 28 × 28 × 256 = 200,704 values → one dense layer with ~200K parameters → high overfitting risk, especially on smaller datasets.
- **Global Average Pooling (GAP)** → average each of the 256 feature channels spatially → a 256-dimensional vector → one dense layer with ~256 parameters.

GAP also has a useful inductive bias: it encourages the network to produce activations that are **spatially distributed** across the patch, rather than relying on features at a single specific location. This is particularly appropriate for histology, where diagnostic features (cell nuclei, gland structure, stroma) can appear anywhere within a 224 × 224 patch.

The `Dropout(0.5)` applied after GAP provides additional regularisation immediately before the final sigmoid output — the point in the network where overfitting is most likely to manifest as overconfident predictions.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from src.models.architectures import get_model

# Build and inspect the model
model = get_model('subtle')
model.summary()


<h2 id="section-10">10. Step 6 — Training</h2>

We train with the following hyperparameters, chosen through experimentation:
- **Optimiser**: Adam with learning rate `1e-5` (very low — prevents training instability)
- **Gradient clipping**: `clipnorm=1.0` (prevents exploding gradients that cause wild validation oscillations)
- **Callbacks**: ModelCheckpoint (save best val_loss), ReduceLROnPlateau (halve LR after 3 stagnant epochs), EarlyStopping (stop after 3 epochs without improvement)

> **Note**: Training requires ~6 minutes per epoch on a T4 GPU. The cells below execute the full training runs. If you want to skip training, pre-trained models can be loaded from the `models/` directory.


In [ ]:
# Configure training
from src.models import run_binary_experiment
from config import DEFAULT_CONFIG

DEFAULT_CONFIG.training.normalise_patches = False
DEFAULT_CONFIG.training.val_max_samples_per_class = 4000

# Uses TRAIN_PATH defined in Section 7 above
TRAIN_DATASET_PATH = TRAIN_PATH

# ============================================================
# EXPERIMENT 2: Normal vs Pure Tumor (sanity check — should be easy)
# ============================================================
print("=" * 60)
print("EXPERIMENT 2: Normal vs Pure Tumor")
print("=" * 60)

exp2_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=2,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp2_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp2_results['results']['auc']:.3f}")


In [ ]:
# ============================================================
# EXPERIMENT 5: Normal vs Boundary (harder)
# ============================================================
print("=" * 60)
print("EXPERIMENT 5: Normal vs Boundary Tumor")
print("=" * 60)

exp5_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=5,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp5_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp5_results['results']['auc']:.3f}")


<h2 id="section-11">11. Step 7 — Test Set Evaluation</h2>

Validation metrics are computed on held-out chunks from the same pool of training slides. The true test of generalisation is the **held-out test set**, which uses entirely different slides not seen during training or validation.

### Design Decision: Pick the Decision Threshold from Validation Data

A sigmoid output is a score, not a calibrated class probability. The default threshold of 0.5 is arbitrary — it is only optimal if the model is perfectly calibrated *and* false positives and false negatives are equally costly. Neither holds in general.

Instead, `find_optimal_threshold()` selects the threshold that maximises **Youden's J statistic** on the validation predictions:

```
J = sensitivity + specificity − 1
  = TPR − FPR
```

Youden's J is maximised at the point on the ROC curve where the trade-off between catching true positives and avoiding false positives is best balanced. The implementation uses scikit-learn's `roc_curve()` to compute J at every candidate threshold and returns the best:

```python
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
j_scores = tpr - fpr
best_threshold = thresholds[np.argmax(j_scores)]
```

**This threshold is selected on validation data only, then held fixed for test evaluation.** Using test labels to select or adjust the threshold would be a form of evaluation leakage — the test set would no longer be truly held out. The threshold is saved in the model's JSON metadata file (`save_model_metadata()`) so it is retrieved consistently whenever the model is loaded:

```python
meta = load_model_metadata('./models/normal_vs_pure_tumor.keras')
threshold = meta['threshold']                     # From validation — not re-fitted
predictions = (test_scores >= threshold).astype(int)
```

### A Critical Bug We Fixed

During development, we discovered that our test evaluation function was feeding **raw, unscaled pixel data** (0–255) to models trained on normalised data (0–1). Validation AUC was 0.93 but test AUC collapsed to 0.53 — essentially random chance.

The fix was simple: apply the same `[0, 1]` scaling and optional per-patch normalisation in the test evaluation path. The lesson was painful but valuable: **preprocessing must be identical between training and inference**, and test evaluation should always be the first sanity check after training, not an afterthought.

In [ ]:
# Test set evaluation
from src.models import evaluate_on_test_set, load_model_metadata

# Uses TEST_PATH defined in Section 7 above

# Load models and metadata
experiments_to_eval = {
    'exp2': {
        'name': 'Normal vs Pure Tumor',
        'model_path': './models/normal_vs_pure_tumor.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['pure_tumor']},
        'results': exp2_results
    },
    'exp5': {
        'name': 'Normal vs Boundary',
        'model_path': './models/normal_vs_boundary.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['boundary_tumor']},
        'results': exp5_results
    },
    'exp3': {
        'name': 'Slide Context Detection',
        'model_path': './models/slide_context_detection.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['normal_from_tumor']},
        'results': exp3_results
    }
}

test_results = {}
for key, exp in experiments_to_eval.items():
    print(f"\n{'='*60}")
    print(f"TEST: {exp['name']}")
    print(f"{'='*60}")

    model = keras.models.load_model(exp['model_path'])
    meta = load_model_metadata(exp['model_path'])
    normalise = meta.get('normalise_patches', False)

    result = evaluate_on_test_set(
        model, TEST_PATH, exp['mapping'], key,
        threshold=meta['threshold'],
        normalise=normalise
    )
    test_results[key] = result

    print(f"Val AUC:  {exp['results']['results']['auc']:.3f}")
    print(f"Test AUC: {result['auc']:.3f}")
    print(f"Gap:      {exp['results']['results']['auc'] - result['auc']:.3f}")
    print(result['report'])


<h2 id="section-12">12. Results</h2>

Let's compare validation and test performance across all experiments.


In [ ]:
# Summary comparison chart
exp_names = ['Exp 2: Normal vs\nPure Tumor', 'Exp 5: Normal vs\nBoundary', 'Exp 3: Slide\nContext']
exp_keys = ['exp2', 'exp5', 'exp3']

val_aucs = [experiments_to_eval[k]['results']['results']['auc'] for k in exp_keys]
test_aucs = [test_results[k]['auc'] for k in exp_keys]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(exp_names))
width = 0.35

# AUC comparison
bars1 = axes[0].bar(x - width/2, val_aucs, width, label='Validation', color='steelblue')
bars2 = axes[0].bar(x + width/2, test_aucs, width, label='Test', color='darkorange')
axes[0].set_ylabel('AUC', fontsize=12)
axes[0].set_title('Validation vs Test AUC', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(exp_names, fontsize=10)
axes[0].legend(fontsize=11)
axes[0].set_ylim(0.4, 1.0)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
for i, (v, t) in enumerate(zip(val_aucs, test_aucs)):
    axes[0].text(i - width/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[0].text(i + width/2, t + 0.02, f'{t:.3f}', ha='center', fontsize=9, fontweight='bold')

# Val-Test gap
gaps = [v - t for v, t in zip(val_aucs, test_aucs)]
colours = ['forestgreen' if g < 0.05 else 'darkorange' if g < 0.1 else 'firebrick' for g in gaps]
axes[1].bar(x, gaps, color=colours, width=0.5)
axes[1].set_ylabel('AUC Gap (Val - Test)', fontsize=12)
axes[1].set_title('Generalisation Gap', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(exp_names, fontsize=10)
axes[1].axhline(y=0, color='black', linewidth=0.5)
for i, g in enumerate(gaps):
    axes[1].text(i, g + 0.005, f'{g:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


<h2 id="section-13">13. Investigating the Field Cancerisation Hypothesis</h2>

The most scientifically interesting result is Experiment 3: can a model detect whether normal-looking tissue came from a slide that also contains a tumor?

Our initial validation results suggested a weak but detectable signal (AUC ~0.64). However, evaluation on the held-out test set showed performance near random chance (AUC ~0.54). To ensure this wasn't a model capacity issue, we tested four architectures of increasing complexity:

| Architecture | Val AUC | Test AUC | Trainable Params |
|---|---|---|---|
| `subtle` (custom CNN) | 0.585 | 0.543 | ~390K |
| `attention` (spatial attention) | 0.610 | 0.467 | ~390K |
| `transfer` (frozen MobileNetV2) | 0.504 | — | ~1.3K |
| `transfer_finetune` (fine-tuned MobileNetV2) | 0.585 | — | ~700K |

The consistency across architectures is telling: the problem isn't model capacity. The weak validation signal likely reflects **slide-level confounds** (staining batch effects, tissue preparation differences) rather than genuine field cancerisation. These confounds are shared between training and validation slides (from the same pool) but don't transfer to the test slides.

### What Would It Take?

Properly testing this hypothesis would require approaches beyond patch-level classification:
- **Multi-instance learning**: aggregate evidence across many patches per slide, rather than classifying patches independently
- **Pathology foundation models**: use feature extractors pre-trained on millions of pathology images (e.g., UNI, CONCH) rather than ImageNet
- **Slide-level prediction**: treat each slide as a single example, using all its patches collectively

This is itself a valuable finding: a clean negative result that establishes what *doesn't* work and points toward what might.

<h2 id="section-14">14. Lessons Learned</h2>

### Technical Lessons

1. **Preprocessing consistency is everything.** Our most dramatic bug was a mismatch between training and test preprocessing: models saw `[0, 1]` scaled data during training but raw `[0, 255]` data during evaluation. Validation AUC was 0.93; test AUC was 0.53. Always have a single source of truth for preprocessing transformations.

2. **Memory management dominates development time.** Whole slide images, patch datasets, and TensorFlow data pipelines all compete for RAM. The most impactful fixes weren't algorithmic — they were setting `num_parallel_calls=2` instead of `AUTOTUNE`, limiting validation set size, and using generators instead of loading data into memory.

3. **Gradient clipping matters more than architecture changes.** Our initial training runs showed wild validation accuracy oscillations (±20% between epochs). Gradient clipping (`clipnorm=1.0`) combined with a low learning rate (`1e-5`) was more effective than any architectural modification.

4. **Start with the easiest experiment.** We used Normal vs Pure Tumor (the visually obvious case) to validate the entire train→evaluate→test pipeline before investing GPU time on harder tasks. This caught the preprocessing bug early.

### Scientific Lessons

5. **Negative results are results.** Our field cancerisation experiment didn't find a generalisable signal, despite trying four architectures. This is informative: it constrains what's possible with patch-level H&E classification and motivates alternative approaches.

6. **Validation ≠ generalisation.** Even with no slide leakage, validation and test performance can diverge significantly when the task is hard. Validation slides share distributional properties with training slides; test slides may not.

7. **Know when to stop.** When four architectures all converge on the same weak result, the bottleneck is likely the data or the task, not the model. Throwing more complexity at a data-limited problem is unlikely to help.

<h2 id="section-15">15. Conclusion</h2>

We built a complete pipeline for automated tumor detection in histopathology images: from gigapixel whole slide images through tissue detection, patch extraction, stain normalisation, and CNN-based classification. Our models achieve **0.87–0.89 test AUC** for detecting tumor tissue — a useful screening tool, though short of the 0.925 achieved by the CAMELYON16 winners with larger models and more data.

The field cancerisation hypothesis — that normal tissue near tumors carries detectable molecular changes — remains unresolved by our patch-level approach. The signal we observed in validation did not survive the test set, suggesting that more sophisticated methods (multi-instance learning, pathology foundation models) are needed to investigate this further.

The full codebase, including all modules, configurations, and notebooks, is available on [GitHub](https://github.com/your-username/camelyon16-pathology).

---

*This work was conducted as part of a computational pathology project at [Company Name]. The CAMELYON16 dataset is publicly available through the [Grand Challenge platform](https://camelyon16.grand-challenge.org/).*